First we load the datasets and visualize what they look like to think how to use them. 

In [3]:
#imports
import pandas as pd

In [4]:
#load the cleaned datasets
booking_requests = pd.read_csv('cleaned_data/booking_requests.csv')
cleaners = pd.read_csv('cleaned_data/cleaners.csv')
customers = pd.read_csv('cleaned_data/customers.csv')
marketing_spend = pd.read_csv('cleaned_data/marketing_spend.csv')

In [5]:
#visualize the datasets

booking_requests


,request_id,customer_id,city,request_date,hours,price_eur,status,cleaner_id,rating,fulfilled,canc,month
0,R0000609,C000609,Lyon,2025-01-14,4,108.0,fulfilled,P00037,5.0,True,False,2025-01-01
1,R0010556,C002995,Bordeaux,2025-05-07,3,81.0,fulfilled,P00186,4.0,True,False,2025-05-01
2,R0001916,C001191,Lyon,2025-02-20,3,81.0,unfulfilled,NaN,NaN,False,False,2025-02-01
3,R0061885,C009288,Paris,2026-01-14,3,81.0,fulfilled,P00212,3.0,True,False,2026-01-01
4,R0015685,C000933,Paris,2025-07-12,3,81.0,fulfilled,P00136,4.0,True,False,2025-07-01
...,...,...,...,...,...,...,...,...,...,...,...,...
132159,R0038441,C004736,Lyon,2025-10-24,4,108.0,fulfilled,P00234,5.0,True,False,2025-10-01
132160,R0040942,C009999,Bordeaux,2025-10-25,3,81.0,fulfilled,P00170,5.0,True,False,2025-10-01
132161,R0027337,C002467,Paris,2025-09-21,2,54.0,fulfilled,P00501,5.0,True,False,2025-09-01
132162,R0106418,C010209,Paris,2026-05-20,3,81.0,fulfilled,P01096,5.0,True,False,2026-05-01


In [6]:
#visualize the datasets
cleaners

,cleaner_id,application_date,city,acquisition_channel,onboarded_date,deactivated_date
0,P00027,2025-01-01,Paris,Meta Ads,2025-01-19,2025-01-30
1,P00002,2025-01-01,Paris,Meta Ads,2025-01-10,2025-12-31
2,P00028,2025-01-01,Paris,Job boards,2025-01-10,2025-08-24
3,P00014,2025-01-02,Paris,Meta Ads,2025-01-19,NaN
4,P00025,2025-01-04,Paris,Job boards,2025-01-23,2025-10-23
...,...,...,...,...,...,...
1506,P01504,2026-06-29,Lille,Meta Ads,2026-07-12,NaN
1507,P01461,2026-06-29,Bordeaux,Job boards,2026-07-13,NaN
1508,P01506,2026-06-30,Lille,Referral,NaN,NaN
1509,P01453,2026-06-30,Lyon,Job boards,2026-07-20,NaN


In [7]:
customers


,customer_id,signup_date,city,acquisition_channel,year,month,day
0,C000192,2025-01-01,Paris,Google Ads,2025,1,1
1,C000039,2025-01-01,Paris,Google Ads,2025,1,1
2,C000572,2025-01-01,Lyon,Organic,2025,1,1
3,C000504,2025-01-01,Lyon,Meta Ads,2025,1,1
4,C000307,2025-01-01,Paris,Google Ads,2025,1,1
...,...,...,...,...,...,...,...
24627,C024347,2026-06-30,Bordeaux,Meta Ads,2026,6,30
24628,C022608,2026-06-30,Paris,Organic,2026,6,30
24629,C024561,2026-06-30,Lille,Google Ads,2026,6,30
24630,C024436,2026-06-30,Lille,Google Ads,2026,6,30


In [8]:
marketing_spend

,month,city,side,channel,spend_eur
0,2025-01-01,Paris,demand,Google Ads,3920.0
1,2025-01-01,Paris,demand,Meta Ads,2880.0
2,2025-01-01,Paris,supply,Job boards,1620.0
3,2025-01-01,Paris,supply,Meta Ads,480.0
4,2025-01-01,Lyon,demand,Google Ads,2290.0
...,...,...,...,...,...
238,2026-06-01,Bordeaux,supply,Meta Ads,1750.0
239,2026-06-01,Lille,demand,Google Ads,2450.0
240,2026-06-01,Lille,demand,Meta Ads,1440.0
241,2026-06-01,Lille,supply,Job boards,770.0


The first thing we remark is that there are two funnels, one for the client, one for the cleaners.
Let's first work on the acquisition of both.

I- Acquisition 

A- Customers


In [42]:
#calculate the number of customers by acquisition channel
customers.groupby('acquisition_channel').size().reset_index()

,acquisition_channel,0
0,Google Ads,8589
1,Meta Ads,7362
2,Organic,6177
3,Referral,2504


What are the kind of marketings spend ?

In [40]:
marketing_spend.groupby("channel").sum().reset_index()[['channel', 'spend_eur']].sort_values(by='spend_eur', ascending=False)   

,channel,spend_eur
0,Google Ads,205420.0
2,Meta Ads,172340.0
1,Job boards,68440.0


B- Cleaners

In [10]:
#calculate the number of cleaners by acquisition channel
cleaners.groupby('acquisition_channel').size().reset_index(name='count').sort_values(by='count', ascending=False)

,acquisition_channel,count
0,Job boards,818
1,Meta Ads,488
2,Referral,205


II- Engaging 

A- Clients

In [11]:
#number of bookings by customer
booking_requests.groupby('customer_id').size().reset_index(name='count').sort_values(by='count', ascending=False)

,customer_id,count
817,C000818,44
135,C000136,44
2869,C002870,37
859,C000860,37
1316,C001317,36
...,...,...
24403,C024536,1
24404,C024537,1
24405,C024538,1
24406,C024539,1


In [12]:
#number of customers with no bookings
noBooking = customers[~customers['customer_id'].isin(booking_requests['customer_id'])].shape[0]
print(noBooking)


141


In [13]:
#customers with only uncompleted bookings
customersWithUncompletedBookings = customers[customers['customer_id'].isin(booking_requests[booking_requests['status'] == 'unfulfilled']['customer_id'])]
customersWithUncompletedBookings.shape[0]


9196

In [14]:
#Do those customers have any completed bookings?
customersWithUncompletedBookings[customersWithUncompletedBookings['customer_id'].isin(booking_requests[booking_requests['status'] == 'completed']['customer_id'])].shape[0]

0

We remark that there is a small number of customers which do not do any booking and a huge number of them who try to do a booking but it is not fullfilled, they are not activated...

In [15]:
#calculate the pourcentage of activation
pourcentage_activation = (customers.shape[0]- noBooking - customersWithUncompletedBookings.shape[0]) / customers.shape[0] * 100
print(f"Pourcentage of activation: {pourcentage_activation:.2f}%")

Pourcentage of activation: 62.09%


We define activation as the successful completion of a customer's first booking. Customers with no booking or only unfulfilled booking attempts are therefore considered non-activated.
Around 62% of the clients activate the service. 

We would now like to know which part of them is kept (retention).

The first step is to keep only the transaction which have succeeded. 

In [16]:
fullfiled_requests = booking_requests[booking_requests['status'] == 'fulfilled']
fullfiled_requests.groupby('customer_id').size().reset_index(name='count').sort_values(by='count', ascending=False)

,customer_id,count
649,C000818,42
127,C000136,39
1363,C001632,34
689,C000860,34
2103,C002573,33
...,...,...
20569,C024587,1
20570,C024588,1
20571,C024589,1
20572,C024591,1


In [17]:
#get the number of clients for each number of fulfilled bookings
fulfilled_bookings_count = fullfiled_requests.groupby('customer_id').size().reset_index(name='count')
fulfilled_bookings_distribution = fulfilled_bookings_count.groupby('count').size().reset_index(name='number_of_customers').sort_values(by='count', ascending=False)
fulfilled_bookings_distribution

,count,number_of_customers
35,42,1
34,39,1
33,34,2
32,33,5
31,32,7
30,31,6
29,30,12
28,29,11
27,28,18
26,27,23


For the customers with only 1 command, what is the reason? Is the last command recent?

In [18]:
#check the date of the last booking for clients with only one fulfilled booking
last_booking_date = fullfiled_requests[fullfiled_requests['customer_id'].isin(fulfilled_bookings_count[fulfilled_bookings_count['count'] == 1]['customer_id'])].groupby('customer_id').max()
last_booking_date

,request_id,city,request_date,hours,price_eur,status,cleaner_id,rating,fulfilled,canc,month
customer_id,,,,,,,,,,,
C000007,R0000007,Paris,2025-01-21,3,81.0,fulfilled,P00020,2.0,True,False,2025-01-01
C000011,R0000011,Paris,2025-01-05,2,54.0,fulfilled,P00001,3.0,True,False,2025-01-01
C000015,R0000015,Paris,2025-01-03,2,54.0,fulfilled,P00003,5.0,True,False,2025-01-01
C000019,R0000019,Paris,2025-01-19,3,81.0,fulfilled,P00006,5.0,True,False,2025-01-01
C000033,R0000033,Paris,2025-01-13,2,54.0,fulfilled,P00007,2.0,True,False,2025-01-01
...,...,...,...,...,...,...,...,...,...,...,...
C024627,R0132300,Lille,2026-06-20,4,108.0,fulfilled,P01389,2.0,True,False,2026-06-01
C024628,R0132301,Lille,2026-06-15,3,81.0,fulfilled,P01393,5.0,True,False,2026-06-01
C024630,R0132303,Lille,2026-06-28,3,81.0,fulfilled,P01080,4.0,True,False,2026-06-01


In [19]:
#count the number of clients with only one fulfilled booking and a last booking date after 2026-05-01
last_booking_date[last_booking_date['month'] > '2026-05-01'].shape[0]


1619

In [20]:
#count the number of clients with only one fulfilled booking and a last booking date after 2026-06-01
last_booking_date[last_booking_date['month'] > '2026-06-01'].shape[0]

0

We can consider that since all may 2026 first time clients did not book anything in june, and that no one had its first booking in june, that those clients are lost.

In [21]:
#calculate proportion of retended customers
proportion_retained = fulfilled_bookings_distribution[fulfilled_bookings_distribution['count'] > 1]['number_of_customers'].sum() / fulfilled_bookings_distribution["number_of_customers"].sum() * 100
print(proportion_retained)

67.72142683814609


In [65]:
#number or retained customer
fulfilled_bookings_distribution[fulfilled_bookings_distribution['count'] > 1]['number_of_customers'].sum()

np.int64(13954)

In [60]:
#Retention 30 days = customer makes another fulfilled booking within 30 days after their first fulfilled booking.
retention_30_days = 0
for customer_id in fulfilled_bookings_count[fulfilled_bookings_count['count'] > 1]['customer_id']:
    first_booking_date = fullfiled_requests[fullfiled_requests['customer_id'] == customer_id].sort_values(by='month').iloc[0]['month']
    second_booking_date = fullfiled_requests[fullfiled_requests['customer_id'] == customer_id].sort_values(by='month').iloc[1]['month']
    if (pd.to_datetime(second_booking_date) - pd.to_datetime(first_booking_date)).days <= 30:
        retention_30_days += 1
print(f"Retention 30 days: {retention_30_days} customers")

Retention 30 days: 5045 customers


In [66]:
#Retention 60 days = customer makes another fulfilled booking within 60 days after their first fulfilled booking.
retention_60_days = 0
for customer_id in fulfilled_bookings_count[fulfilled_bookings_count['count'] > 1]['customer_id']:
    first_booking_date = fullfiled_requests[fullfiled_requests['customer_id'] == customer_id].sort_values(by='month').iloc[0]['month']
    second_booking_date = fullfiled_requests[fullfiled_requests['customer_id'] == customer_id].sort_values(by='month').iloc[1]['month']
    if (pd.to_datetime(second_booking_date) - pd.to_datetime(first_booking_date)).days <= 60:
        retention_60_days += 1
print(f"Retention 60 days: {retention_60_days} customers")

Retention 60 days: 11757 customers


Let's try to find why clients do not continue to use the service. To find patterns, we will try to find the avaluation of the last transaction  of every client who did not do a new command the last month. 

In [25]:
#extract the evaluation of the last command for each customer if there is no reservation after may 2026
last_evaluation = fullfiled_requests.groupby('customer_id').last().reset_index()[['customer_id', 'rating']]
last_evaluation_after_may_2026 = last_evaluation[last_evaluation['customer_id'].isin(last_booking_date[last_booking_date['month'] <= '2026-05-01'].index)]
last_evaluation_after_may_2026.groupby('rating').size().reset_index(name='count').sort_values(by='count', ascending=False)

,rating,count
4,5.0,1631
2,3.0,1416
3,4.0,793
1,2.0,728
0,1.0,464


In [29]:
#same for clients with more than one fulfilled booking
last_evaluation_multiple_bookings = fullfiled_requests[fullfiled_requests['customer_id'].isin(fulfilled_bookings_count[fulfilled_bookings_count['count'] > 1]['customer_id'])].groupby('customer_id').last().reset_index()[['customer_id', 'rating']]
last_evaluation_multiple_bookings.groupby('rating').size().reset_index(name='count').sort_values(by='count', ascending=False)

,rating,count
4,5.0,7786
3,4.0,3788
2,3.0,1176
1,2.0,752
0,1.0,452


The evaluation seems not to explain why a client leave the service.

In [34]:
#check the number of cancelled bookings
canceled_bookings_count = booking_requests[booking_requests['canc'] == True]
print(f"Number of canceled bookings: {canceled_bookings_count.shape[0]}")
canceled_bookings_count

Number of canceled bookings: 6505


,request_id,customer_id,city,request_date,hours,price_eur,status,cleaner_id,rating,fulfilled,canc,month
14,R0079097,C013039,Bordeaux,2026-02-20,4,108.0,cancelled_by_customer,NaN,NaN,False,True,2026-02-01
15,R0051618,C006311,Paris,2025-12-15,2,54.0,cancelled_by_customer,NaN,NaN,False,True,2025-12-01
77,R0046159,C002745,Lyon,2025-11-26,2,54.0,cancelled_by_customer,NaN,NaN,False,True,2025-11-01
127,R0121289,C015447,Paris,2026-06-21,2,54.0,cancelled_by_customer,NaN,NaN,False,True,2026-06-01
151,R0022467,C004437,Paris,2025-08-19,3,81.0,cancelled_by_customer,NaN,NaN,False,True,2025-08-01
...,...,...,...,...,...,...,...,...,...,...,...,...
132029,R0067209,C003048,Bordeaux,2026-01-30,2,54.0,cancelled_by_customer,NaN,NaN,False,True,2026-01-01
132106,R0029806,C006799,Paris,2025-09-03,4,108.0,cancelled_by_customer,NaN,NaN,False,True,2025-09-01
132110,R0084661,C015081,Paris,2026-03-13,3,81.0,cancelled_by_customer,NaN,NaN,False,True,2026-03-01
132133,R0059395,C000860,Paris,2026-01-14,2,54.0,cancelled_by_customer,NaN,NaN,False,True,2026-01-01


In [ ]:
#check if cancelled booking are considered as unfulfilled bookings
canceled_bookings_count[canceled_bookings_count['fulfilled'] == True].shape[0]

6505

Hopefully there are no reservation marked as fullfilled but cancelled (thanks to the person who cleaned the dataset ;))

In [38]:
#check the price of booking per hours
booking_requests['price_per_hour'] = booking_requests['price_eur'] / booking_requests['hours']
booking_requests['price_per_hour'].describe()

count    132164.0
mean         27.0
std           0.0
min          27.0
25%          27.0
50%          27.0
75%          27.0
max          27.0
Name: price_per_hour, dtype: float64

The compagny does not apply any reduction.

Let's focus now on the refferal

In [45]:
#number of client gained with referral program
referral_customers = customers[customers['acquisition_channel'] == 'Referral']
referral_customers.shape[0]

2504

Let's find the number of active customers gained with refferal and associate them with the number of active clients.

In [46]:
#number of active clients gained with referral program
#customers who have made at least one fulfilled booking
active_referral_customers = referral_customers[referral_customers['customer_id'].isin(fullfiled_requests['customer_id'])]
active_referral_customers.shape[0]

2104

In [62]:
#number of client gained by referal compared to the total number of clients
proportion_referral_customers =  referral_customers.shape[0]/customers.shape[0]
print(f"Part of clients acquired by referral: {proportion_referral_customers*100:.2f}%")


Part of clients acquired by referral: 10.17%


In [59]:
#number of client gained with referral by month and year
refferal_customers_by_month_and_year = referral_customers.groupby(['month', "year"]).size().reset_index(name='count').sort_values(by='year', ascending=True)
refferal_customers_by_month_and_year

,month,year,count
0,1,2025,67
2,2,2025,72
6,4,2025,97
4,3,2025,75
12,7,2025,105
13,8,2025,119
10,6,2025,96
8,5,2025,113
14,9,2025,151
15,10,2025,148


There is an acceleration of the customers recruited by referal in 2026. 

In [ ]:
#are clients gained by referral program more likely to be retained than clients gained by other acquisition channels?
#calculate the proportion of retained clients for referral customers
# Clients ayant au moins 1 booking fulfilled
active_customers = (
    fullfiled_requests['customer_id']
    .dropna()
    .unique()
)

# Nombre de bookings fulfilled par client
fulfilled_count = (
    fullfiled_requests
    .groupby('customer_id')
    .size()
    .rename('fulfilled_count')
)

# Clients retenus = au moins 2 bookings fulfilled
retained_customers = fulfilled_count[fulfilled_count > 1].index

# On part uniquement des clients ayant au moins 1 booking fulfilled
active = customers[
    customers['customer_id'].isin(active_customers)
].copy()

active['retained'] = active['customer_id'].isin(retained_customers)

retention_by_channel = (
    active
    .groupby('acquisition_channel')['retained']
    .agg(['sum', 'count'])
)

retention_by_channel['retention_rate'] = (
    retention_by_channel['sum']
    / retention_by_channel['count']
    * 100
)

retention_by_channel

,sum,count,retention_rate
acquisition_channel,,,
Google Ads,4853,7229,67.132383
Meta Ads,4205,6167,68.185503
Organic,3438,5105,67.345739
Referral,1458,2104,69.296578


Referral isn't more efficient than other acquisition methods.

In [55]:
#let's calculate the average revenue per customer
revnuePerCustomer = fullfiled_requests.groupby('customer_id')['price_eur'].sum().reset_index(name='revenue')
mean_revenue_per_customer = revnuePerCustomer['revenue'].mean()
print(f"Average revenue per customer: {mean_revenue_per_customer:.2f}€")

Average revenue per customer: 384.84€


In [64]:
#revenue per customer by acquisition channel
revenue_per_customer_by_channel = fullfiled_requests.merge(customers[['customer_id', 'acquisition_channel']], on='customer_id').groupby('acquisition_channel')['price_eur'].sum().reset_index(name='total_revenue')
revenue_per_customer_by_channel = revenue_per_customer_by_channel.merge(customers.groupby('acquisition_channel').size().reset_index(name='number_of_customers'), on='acquisition_channel')
revenue_per_customer_by_channel['average_revenue_per_customer'] = revenue_per_customer_by_channel['total_revenue'] / revenue_per_customer_by_channel['number_of_customers']
revenue_per_customer_by_channel

,acquisition_channel,total_revenue,number_of_customers,average_revenue_per_customer
0,Google Ads,2745468.0,8589,319.649319
1,Meta Ads,2395035.0,7362,325.323961
2,Organic,1963062.0,6177,317.801846
3,Referral,826119.0,2504,329.919728


No loops were identified.
The value lack at different points :
- referral is not clearly evaluated (what is the number of clients who gave a referral)
- Retention rate is pretty good (67%) compared to activation rate (62%), a solution should be found to activate users who smay be satisfied
- Unfullfilled reservation are the main reasons : they explain the low activation rate. Identifying the patterns behind those fallback may help the company
- Acquisition path are equivalent in terms of rentention and revenue.